# Оценка клиники по тексту отзыва

Два класса: 1–2 звезды и 4–5 звёзд. Сравним логистическую регрессию и нейросеть на PyTorch. Для теста возьмём отзывы о клиниках, которых не было в обучении.

In [22]:
from collections import Counter
from itertools import combinations
from pathlib import Path
import hashlib
import inspect
import json
import subprocess

import pandas as pd
import torch
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from torch import nn
from torch.utils.data import DataLoader

from model import LABELS, UNK, ReviewClassifier, encode, pack_reviews, tokenize

SEED = 42
torch.manual_seed(SEED)
torch.set_num_threads(1)
artifacts = Path("artifacts")
artifacts.mkdir(exist_ok=True)

## Данные

Файл Yandex Geo Reviews Dataset 2023 скачивается один раз. Контрольная сумма SHA-256 нужна, чтобы при повторном запуске получить ту же выборку.

In [23]:
revision = "02366bc66bd43aa69e707747487ec8f3c22cac84"
filename = "train-00000-of-00001-49261e4e5a35a5a0.parquet"
url = f"https://huggingface.co/datasets/d0rj/geo-reviews-dataset-2023/resolve/{revision}/data/{filename}?download=true"
expected_sha = "10dee53b12dd9dc99b499f9d3fdc0617fd46ac3ebd594654fa5e89e0314a040f"
path = Path("data/reviews.parquet")
path.parent.mkdir(exist_ok=True)

def verify_file(candidate):
    with candidate.open("rb") as stream:
        checksum = hashlib.file_digest(stream, "sha256").hexdigest()
    if checksum != expected_sha:
        raise ValueError(f"Повреждён файл {candidate}. Удалите его и повторите загрузку.")

if path.exists():
    verify_file(path)
else:
    partial = path.with_suffix(".part")
    subprocess.run(["curl", "-fsSL", "--retry", "3", "-o", str(partial), url], check=True)
    verify_file(partial)
    partial.replace(path)

raw = pd.read_parquet(path)
print(f"Исходных строк: {len(raw):,}")

Исходных строк: 500,000


In [24]:
tags = raw.rubrics.fillna("").str.split(";").explode().str.strip()
frequency = tags[tags.ne("")].value_counts()
print("Различных рубрик:", len(frequency))
print("Рубрики со словами, связанными с клиниками и стоматологией:")
display(frequency[frequency.index.str.contains("клиник|стомат", case=False)].rename("отзывы").to_frame())

Различных рубрик: 1370
Рубрики со словами, связанными с клиниками и стоматологией:


,отзывы
rubrics,
"Медцентр, клиника",8231
Стоматологическая клиника,6928
Ветеринарная клиника,3169
Стоматологическая поликлиника,2333
Поликлиника для взрослых,2221
Детская поликлиника,1386
Гинекологическая клиника,1008
Наркологическая клиника,181
Стоматологические материалы и оборудование,81


У одной организации бывает несколько рубрик. Достаточно совпадения с одной из девяти выбранных.

In [25]:
selected = [
    "Медцентр, клиника", "Стоматологическая клиника", "Стоматологическая поликлиника",
    "Поликлиника для взрослых", "Детская поликлиника", "Гинекологическая клиника",
    "Наркологическая клиника", "Детская клиника", "Психиатрическая клиника",
]
display(frequency.loc[selected].rename("отзывы с рубрикой").to_frame())
source_ids = tags.index[tags.isin(selected)].unique()
reviews = raw.loc[source_ids, ["name_ru", "address", "rating", "text"]].copy()
del raw, tags
print("После фильтра по рубрикам:", len(reviews))
display(reviews.rating.value_counts(dropna=False).sort_index().rename("отзывы").to_frame())

,отзывы с рубрикой
rubrics,
"Медцентр, клиника",8231
Стоматологическая клиника,6928
Стоматологическая поликлиника,2333
Поликлиника для взрослых,2221
Детская поликлиника,1386
Гинекологическая клиника,1008
Наркологическая клиника,181
Детская клиника,73
Психиатрическая клиника,41


После фильтра по рубрикам: 17757


,отзывы
rating,
1,1843
2,448
3,600
4,827
5,14039


## Очистка и разбиение

При поиске повторов игнорируем регистр и пунктуацию, «ё» заменяем на «е». Тем же способом приводим к общему виду названия и адреса клиник.

In [26]:
def normalize(series):
    return (series.fillna("").str.normalize("NFKC").str.casefold().str.replace("ё", "е")
            .str.replace(r"[\W_]+", " ", regex=True).str.strip())

reviews = reviews[reviews.rating.isin([1, 2, 4, 5])].copy()
reviews["label"] = (reviews.rating >= 4).astype(int)
reviews["text_key"] = normalize(reviews.text)
name, address = normalize(reviews.name_ru), normalize(reviews.address)
usable = name.ne("") & address.ne("") & reviews.text_key.ne("")
print("Без текста, названия или адреса:", (~usable).sum())
reviews["clinic"] = name + " | " + address
reviews = reviews.loc[usable].copy()
conflicting = reviews.groupby("text_key").label.transform("nunique").gt(1)
print("Строк с противоречивой разметкой:", conflicting.sum())
reviews = reviews.loc[~conflicting]
print("Лишних копий текста:", reviews.text_key.duplicated().sum())
reviews = reviews.drop_duplicates("text_key")
print("Осталось отзывов:", len(reviews))

Без текста, названия или адреса: 14
Строк с противоречивой разметкой: 0
Лишних копий текста: 0
Осталось отзывов: 17143


Все отзывы с одинаковыми названием и адресом идут в одну выборку. Для обучения оставляем 70% клиник, остальные делим поровну между валидацией и тестом.

In [27]:
def split_clinics(frame, held_out_fraction):
    splitter = GroupShuffleSplit(n_splits=1, test_size=held_out_fraction, random_state=SEED)
    left, right = next(splitter.split(frame, groups=frame.clinic))
    return frame.iloc[left], frame.iloc[right]

train, held_out = split_clinics(reviews, 0.30)
valid, test = split_clinics(held_out, 0.50)
parts = {"Обучение": train, "Валидация": valid, "Тест": test}
for a, b in combinations(parts.values(), 2):
    assert set(a.clinic).isdisjoint(b.clinic)
    assert set(a.text_key).isdisjoint(b.text_key)
assert sum(map(len, parts.values())) == len(reviews)
summary = []
for part, frame in parts.items():
    counts = frame.label.value_counts()
    summary.append([part, counts[0], counts[1], len(frame), frame.clinic.nunique(), 100 * len(frame) / len(reviews)])
display(pd.DataFrame(summary, columns=["Часть", "Низкая", "Высокая", "Отзывы", "Клиники", "% отзывов"])
        .set_index("Часть").round(1))
print("Пересечений по ключам клиник и текстов между частями нет.")

,Низкая,Высокая,Отзывы,Клиники,% отзывов
Часть,,,,,
Обучение,1583,10446,12029,7200,70.2
Валидация,333,2231,2564,1543,15.0
Тест,374,2176,2550,1544,14.9


Пересечений по ключам клиник и текстов между частями нет.


## TF-IDF и логистическая регрессия

«Помогли» и «не помогли» — два разных признака. Низкие оценки встречаются реже, поэтому при обучении обеих моделей ошибка на таком отзыве весит больше.

In [28]:
baseline = make_pipeline(
    TfidfVectorizer(tokenizer=tokenize, token_pattern=None, lowercase=False, ngram_range=(1, 2), min_df=2),
    LogisticRegression(class_weight="balanced", max_iter=500, random_state=SEED),
)
baseline.fit(train.text, train.label)
print("Число признаков TF-IDF:", len(baseline[0].vocabulary_))

Число признаков TF-IDF: 77330


## Нейросеть с EmbeddingBag

Модель учит 64 числа для каждого слова в словаре. EmbeddingBag усредняет векторы слов в отзыве, а линейный слой учится по этому среднему выбирать класс. Порядок слов теряется: у фраз «врач слушает, администратор перебивает» и «администратор слушает, врач перебивает» среднее одинаковое.

In [29]:
counts = Counter(word for text in train.text for word in tokenize(text))
vocab = {UNK: 0}
for word in sorted(counts):
    if counts[word] >= 2:
        vocab[word] = len(vocab)
assert "не" in tokenize("Мне не понравилось") and "не" in vocab
print("Размер словаря с <unk>:", len(vocab))
print(inspect.getsource(ReviewClassifier))

Размер словаря с <unk>: 20924
class ReviewClassifier(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.EmbeddingBag(vocab_size, 64, mode="mean")
        self.linear = nn.Linear(64, 2)

    def forward(self, tokens, offsets):
        vectors = self.embedding(tokens, offsets)
        return self.linear(vectors)



В словарь входят слова, которые встретились в обучении хотя бы дважды. У остальных общий индекс `<unk>`.

In [30]:
def collate(batch):
    reviews_in_batch, labels = zip(*batch)
    tokens, offsets = pack_reviews(reviews_in_batch)
    return tokens, offsets, torch.tensor(labels, dtype=torch.long)

def make_loader(frame, shuffle=False):
    samples = [(encode(text, vocab), label) for text, label in zip(frame.text, frame.label)]
    return DataLoader(samples, batch_size=64, shuffle=shuffle, collate_fn=collate,
                      generator=torch.Generator().manual_seed(SEED))

@torch.inference_mode()
def predict_batches(model, loader):
    model.eval()
    predictions = []
    for tokens, offsets, _ in loader:
        predictions.extend(model(tokens, offsets).argmax(dim=1).tolist())
    return predictions

train_loader = make_loader(train, shuffle=True)
valid_loader = make_loader(valid)
model = ReviewClassifier(len(vocab))
class_counts = torch.tensor(train.label.value_counts().sort_index().to_numpy(), dtype=torch.float32)
loss_fn = nn.CrossEntropyLoss(weight=len(train) / (2 * class_counts))
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

Сохраняем веса той эпохи, где macro-F1 на валидации выше всего.

In [31]:
history = []
best_f1 = -1
for epoch in range(1, 11):
    model.train()
    for tokens, offsets, labels in train_loader:
        optimizer.zero_grad()
        logits = model(tokens, offsets)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()

    valid_pred = predict_batches(model, valid_loader)
    score = f1_score(valid.label, valid_pred, average="macro")
    history.append((epoch, score))
    if score > best_f1:
        best_f1 = score
        best_epoch = epoch
        torch.save(model.state_dict(), artifacts / "model.pt")

(artifacts / "vocab.json").write_text(json.dumps(vocab, ensure_ascii=False, indent=1), encoding="utf-8")
display(pd.DataFrame(history, columns=["Эпоха", "Macro-F1 на валидации"]).set_index("Эпоха").round(4))
print(f"Лучшая эпоха: {best_epoch}, macro-F1: {best_f1:.4f}")

,Macro-F1 на валидации
Эпоха,
1,0.8839
2,0.9129
3,0.9168
4,0.9100
5,0.9099
6,0.9102
7,0.9093
8,0.9095
9,0.9069


Лучшая эпоха: 3, macro-F1: 0.9168


## Результаты на тесте

Recall показывает, какую долю отзывов с 1–2 звёздами модель верно отнесла к низким оценкам. Precision — как часто прогноз «низкая» совпал с оценкой автора.

In [32]:
model.load_state_dict(torch.load(artifacts / "model.pt", map_location="cpu", weights_only=True))
predictions = {
    "TF-IDF + LogisticRegression": baseline.predict(test.text),
    "Нейросеть с EmbeddingBag": predict_batches(model, make_loader(test)),
}
scores = []
for title, pred in predictions.items():
    scores.append([title, f1_score(test.label, pred, average="macro"),
                   precision_score(test.label, pred, pos_label=0),
                   recall_score(test.label, pred, pos_label=0)])
results = pd.DataFrame(scores, columns=["Модель", "Macro-F1", "Precision низкой", "Recall низкой"])
display(results.set_index("Модель").round(4))

,Macro-F1,Precision низкой,Recall низкой
Модель,,,
TF-IDF + LogisticRegression,0.8911,0.7366,0.9198
Нейросеть с EmbeddingBag,0.9131,0.8101,0.9011


In [33]:
torch_pred = predictions["Нейросеть с EmbeddingBag"]
matrix = confusion_matrix(test.label, torch_pred, labels=[0, 1])
display(pd.DataFrame(matrix, index=pd.Index(LABELS, name="Оценка автора"),
                     columns=pd.Index(LABELS, name="Прогноз нейросети")))
print(f"Из {matrix[0].sum()} отзывов с 1–2 звёздами нейросеть верно классифицировала {matrix[0, 0]}. Остальным {matrix[0, 1]} она предсказала высокую оценку.")

Прогноз нейросети,низкая,высокая
Оценка автора,,
низкая,337,37
высокая,79,2097


Из 374 отзывов с 1–2 звёздами нейросеть верно классифицировала 337. Остальным 37 она предсказала высокую оценку.


## Ошибки нейросети

Первые две ошибки каждого типа по порядку строк в исходном файле.

In [34]:
mistakes = test.assign(prediction=torch_pred).query("label != prediction").sort_index()
examples = mistakes.groupby("label", sort=True).head(2).sort_values("label", kind="stable")
example_table = examples[["rating", "prediction"]].rename(columns={"rating": "Звёзды автора", "prediction": "Прогноз"})
example_table["Прогноз"] = example_table["Прогноз"].map(dict(enumerate(LABELS)))
display(example_table.rename_axis("Строка в Parquet"))

,Звёзды автора,Прогноз
Строка в Parquet,,
3277,1,высокая
13830,1,высокая
2578,5,низкая
16000,4,низкая


| Строка | Оценка → прогноз | Что написал автор | Что могло сбить модель |
|---|---|---|---|
| 3277 | 1 → высокая | В регистратуре нагрубили. Возвращаться не хочется. | Модель могла упустить отрицание во фразе о повторном визите. |
| 13830 | 1 → высокая | Интерьер и приём понравились. Результатом лечения автор разочарован. | Похвала в начале могла перевесить жалобу. |
| 2578 | 5 → низкая | Прошлый массаж оставил синяки, а курс в этом центре понравился. | Жалобу на прошлый опыт легко принять за жалобу на этот центр. |
| 16000 | 4 → низкая | В поликлинике чисто, но консультацией автор недоволен. | Жалоба занимает почти весь отзыв. По тексту трудно ожидать четыре звезды. |

## Итог

Логистическая регрессия верно распознала больше низких оценок (344 против 337). Нейросеть реже принимала высокую оценку за низкую (79 раз против 123), и её macro-F1 оказался выше (0,9131 против 0,8911).